Urdu OCR Project

Code Saviours SI-26

Name: Eman Fatima
Week: 2
Date: 9 July 2026

This notebook preprocesses the Week 1 Urdu OCR dataset (loaded from GitHub)
and tests baseline performance using Tesseract OCR to identify why a custom
model is needed for Urdu text recognition.

In [ ]:
!git clone https://github.com/emanfatimaa05/urdu-ocr-codesaviours-si26-eman.git

In [ ]:
import shutil
import os

os.makedirs('data/raw', exist_ok=True)
shutil.copytree('urdu-ocr-codesaviours-si26-eman/data/raw', 'data/raw', dirs_exist_ok=True)

print('Copied successfully!')

In [ ]:
import glob
images = glob.glob('data/raw/**/*.jpg', recursive=True) + glob.glob('data/raw/**/*.png', recursive=True)
print(f'Found {len(images)} images')
print(images[:5])

In [ ]:
!pip install opencv-python-headless pillow
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
print('Libraries loaded successfully!')

In [ ]:
def preprocess_image(image_path, save_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (512, 128))
    denoised = cv2.fastNlMeansDenoising(resized, h=10)
    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)
    cv2.imwrite(save_path, binary)
    return binary

os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')

In [28]:
import glob

all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)
print(f'Found {len(all_images)} images to process')

processed_count = 0
for img_path in all_images:
    filename = os.path.basename(img_path)
    save_path = f'data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder')

Found 103 images to process
Done! Processed 103 images
Check data/processed/ folder


In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
from PIL import Image

test_images = list(glob.glob('data/processed/*.png'))[:5]

print('=== Tesseract Results on Urdu Images ===')
print()
for img_path in test_images:
    img = Image.open(img_path)
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {img_path}')
    print(f'Tesseract output: {result}')
    print('---')

In [ ]:
import pandas as pd
labels = pd.read_csv('urdu-ocr-codesaviours-si26-eman/data/labels.csv.')
labels.head(10)

In [ ]:
test_filenames = ['book (26).png', 'news (20).png', 'book (15).png', 'screenshot (15).png']

for fname in test_filenames:
    match = labels[labels['image'].str.contains(fname, regex=False, na=False)]
    print(match)
    print('---')

In [ ]:
remaining = ['book (23).png', 'book (11).png', 'news (15).png']

for fname in remaining:
    match = labels[labels['image'].str.contains(fname, regex=False, na=False)]
    print(match)
    print('---')

Gap Analysis: Tesseract OCR on Urdu Images
Image 1: book (23).png
Actual text: ہوٹل کا کھانا معیاری اور بہت مزے کا تھا۔
Tesseract output: (empty)
What went wrong: No output at all — complete failure to detect any text.
Image 2: news (20).png
Actual text: 73 سالہ شبیر احمد... (long paragraph)
Tesseract output: garbled text with wrong characters
What went wrong: Attempted to read the text and got the general shape right, but substituted wrong letters throughout, especially where Urdu letters connect/join.
Image 3: book (11).png
Actual text: ہوٹل کے واش روم گندے تھے۔ ہوٹل کی صفائی...
Tesseract output: (empty)
What went wrong: No output — same complete failure.
Image 4: screenshot (15).png
Actual text: زبان اردوئے معلّٰی
Tesseract output: (empty)
What went wrong: No output at all — screenshots may have background clutter or unusual fonts adding difficulty.
Image 5: news (15).png
Actual text: انہوں نے کہا کہ وفاق کے جل اصلاحات کے لیے صوبوں...
Tesseract output: mostly gibberish/disconnected letters, no coherent words
What went wrong: Output looks like shapes were detected but not mapped into meaningful Urdu words.
Summary
Tesseract fails on Urdu because the script is cursive, meaning letters change shape depending on their position and connect fluidly to neighboring letters — this makes it hard for Tesseract's default segmentation to correctly separate individual characters. Out of 5 test images, 3 produced no output whatsoever, and the 2 that did produce output were filled with incorrect or scrambled characters rather than accurate text. This confirms the need for a custom-trained OCR model specifically built for Urdu.